# Hybrid quantum PPO on a real IBM Quantum QPU

Notebook version of `training_quantumPPO_hardware.py`.

This notebook does **not** change the simulator trainer. It loads the code of `training_quantumPPO_fast.ipynb` and swaps only the circuit back-end (`PQCSimulator` → `HardwarePQC`). Pre-encoder, actor/critic heads, parameter-shift autograd, PPO loss, parallel envs, the `.pt` checkpoint and the CSV / `run_info.json` files are all the same code, so the learning curves continue across:

1. train on the simulator (`training_quantumPPO_fast.ipynb`)
2. fine-tune on hardware (this notebook, `RESUME = "results/qppo_uav_traj_model.pt"`)
3. go back to the simulator (fast notebook, `RESUME = "results_hw/qppo_uav_traj_model.pt"`)

Workflow: `QiskitRuntimeService` → backend (least busy or by name) → `generate_preset_pass_manager(optimization_level=3)` → ISA circuit → Runtime V2 primitive `.run(pubs)`.

**Try it free first:** set `FAKE_BACKEND = "FakeFez"` in section 6 for a local dry run with that device's noise (no account, no QPU time).

## 1. Imports and the simulator trainer
The fast trainer's code cells (sections 1 to 7) are executed into a module called `qppo`, so the hooks below can swap parts of it. Its demo, run-settings and training cells are skipped. If only `training_quantumPPO_fast.py` is available, it is imported instead.

In [ ]:
import os
import json
import time
import types
import contextlib
import warnings
# local fake-backend runs ignore error mitigation; the real service applies it, so silence the repeat warning
warnings.filterwarnings("ignore", message="The resilience_level option has no effect in local testing mode")
import numpy as np
import torch

from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager


def _exec_notebook(path, ns, skip_tags=()):
    """Run the code cells of a notebook into the dict ns (IPython magics are dropped)."""
    with open(path) as f:
        nb = json.load(f)
    for cell in nb["cells"]:
        if cell["cell_type"] != "code" or set(cell.get("metadata", {}).get("tags", [])) & set(skip_tags):
            continue
        src = "".join(cell["source"])
        src = "\n".join(l for l in src.splitlines() if not l.lstrip().startswith(("%", "!")))
        exec(compile(src, path, "exec"), ns)


def load_fast_trainer():
    if not os.path.exists("training_quantumPPO_fast.ipynb"):
        import training_quantumPPO_fast as mod
        return mod
    mod = types.ModuleType("training_quantumPPO_fast")
    ns = mod.__dict__
    ns["display"] = print
    if os.path.exists("UAV_traj_env.py"):
        from UAV_traj_env import SystemConfig, UAVTrajectoryEnv
        ns.update(SystemConfig=SystemConfig, UAVTrajectoryEnv=UAVTrajectoryEnv)
    else:
        ns["_SKIP_ENV_DEMO"] = True
        _exec_notebook("UAV_traj_env.ipynb", ns)
    _exec_notebook("training_quantumPPO_fast.ipynb", ns, skip_tags=("env", "demo", "run"))
    return mod


qppo = load_fast_trainer()
# originals, kept before the hooks in section 5 replace them
_OrigPQCSimulator = qppo.PQCSimulator
_OrigQuantumPPOAgent = qppo.QuantumPPOAgent
_orig_evaluate = qppo.evaluate
_orig_save_results = qppo.save_results
print("fast trainer loaded:", qppo.QuantumPPOAgent.__name__, "| M =", qppo.SystemConfig().M, "qubits")

## 2. IBM Quantum credentials

The key is read from environment variables, never written in the notebook. Set them before starting Jupyter:

```bash
export QISKIT_IBM_TOKEN="..."      # IBM Cloud API key
export QISKIT_IBM_INSTANCE="..."   # instance CRN
```

If they are not set, the account saved once with `QiskitRuntimeService.save_account(...)` is used. Not needed for a `FAKE_BACKEND` dry run.

In [ ]:
IBM_API_KEY = os.environ.get("QISKIT_IBM_TOKEN", "<YOUR_API_KEY>")
IBM_CRN = os.environ.get("QISKIT_IBM_INSTANCE", "<YOUR_CRN>")
IBM_CHANNEL = "ibm_quantum_platform"

## 3. Hardware settings

| setting | value | why |
|---|---|---|
| backend | least busy, or e.g. `ibm_fez` | Only $M$ qubits are needed, so every device fits. Heron devices (`ibm_fez`, `ibm_marrakesh`, `ibm_kingston`, `ibm_torino`) have ~2-3× lower 2-qubit errors. Chosen once per run so the noise doesn't change mid-training. |
| optimization_level | 3 | The circuit is transpiled **once** with unbound parameters and reused for every job, so the cost of level 3 is paid once; it gives noise-aware layout and the most gate cancellation. `seed_transpiler=1234` keeps the layout reproducible across resumes. |
| primitive | EstimatorV2 | The PQC output is $\langle Z_q\rangle$, which Estimator returns directly, and only Estimator has error mitigation. `"sampler"` gives raw counts. |
| resilience_level | 1 (TREX) | Readout error dominates for this shallow circuit and biases $\langle Z\rangle$ toward 0. Level 2 (ZNE) costs ~3× QPU time and its variance hurts parameter-shift differences. |
| shots | 512 | Shot noise ~0.044, the same order as the error left after TREX. More shots cost QPU time and buy little; below ~256 shot noise dominates. |
| dynamical decoupling | on, XY4 | Free: fills idle windows to cancel dephasing. |
| gate twirling | off | Mainly pays off with ZNE. |
| mode | session (falls back to job) | PPO jobs are sequential, so a session keeps the QPU between jobs instead of re-queuing each one. The Open Plan does not allow sessions; then job mode is used automatically. |
| max_qpu_minutes | 8 | Usage is checked before every job and the run stops cleanly at the budget. |

**PPO changes for hardware** (everything else is the fast trainer's `TRAIN_CFG`): fewer epochs (each epoch re-runs every circuit), large minibatches (each minibatch = 1 forward + 1 parameter-shift job per network), `pqc_grad_samples = 16`, actor and critic circuits of a rollout step in one job, KL from the minibatch log-probs (no extra pass), greedy evaluation on the local ideal simulator, and a checkpoint after every update.

**Cost is set by the number of jobs, not shots.** A real run measured ~27 s of session time per job (submission, compilation, loading and readout calibration dominate). Jobs per update ≈ `rollout_steps / n_envs` (rollout) + 1 (bootstrap) + 2 × `policy_epochs` × minibatches + 2 × `value_epochs` × minibatches. The run prints the measured seconds per job after each update. The intended use is **fine-tuning** a simulator checkpoint for a few updates, not training from scratch.

In [ ]:
HW_CFG = {
    "backend": None,                 # None -> least_busy real device; or e.g. "ibm_fez"
    "fake_backend": None,            # e.g. "FakeFez": local dry run on Aer with that device's noise
    "optimization_level": 3,
    "seed_transpiler": 1234,
    "primitive": "estimator",        # "estimator" (mitigated <Z>) or "sampler" (raw counts)
    "shots": 512,
    "resilience_level": 1,           # estimator only: 0 none, 1 TREX readout mitigation, 2 + ZNE
    "dynamical_decoupling": True,
    "dd_sequence": "XY4",
    "twirl_gates": False,
    "mode": "session",               # "session" | "job"
    "max_qpu_minutes": 8.0,
    "eval_on_hardware": False,
    "job_tags": ["uav_qppo"],
}

HW_TRAIN_OVERRIDES = {
    "policy_epochs": 2,
    "value_epochs": 2,
    "rollout_steps": 3400,
    "mini_batch_size": 680,
    "n_envs": 68,
    "pqc_grad_samples": 16,          # 16 rows per minibatch = same shifted circuits as 2 x 8
    "n_episodes": 200,
    "plot_every": 1,                 # checkpoint + CSVs after every update
    "noise": True,                   # results come from shots on noisy hardware
    "out_dir": "results_hw",
}


class QPUBudgetExceeded(RuntimeError):
    pass

## 4. Hardware back-end
### 4.1 Connection, transpilation, primitive and budget
One `HardwareContext` per run, shared by the actor and critic PQCs (same session, same physical qubits). Sessions are billed by reserved wall time, jobs by quantum seconds; the budget check uses the larger of the two, and the queue wait before the first job is not counted.

In [ ]:
class HardwareContext:
    def __init__(self, hw):
        self.hw = hw
        self.budget_s = float(hw["max_qpu_minutes"]) * 60.0
        self.n_jobs = self.n_circuits = 0
        self.job_seconds = 0.0            # sum of quantum seconds reported by the jobs
        self.session_seconds = 0.0        # wall time of sessions already closed
        self._session_t0 = None
        self._local = 0                   # >0 -> route circuits to the local ideal simulator
        self._isa = {}
        self.session = None
        self.backend = self._get_backend()
        self.mode = "local" if hw["fake_backend"] else hw["mode"]
        self._open()

    # ---- connect ----
    def _get_backend(self):
        hw = self.hw
        if hw["fake_backend"]:
            from qiskit_ibm_runtime import fake_provider
            be = getattr(fake_provider, hw["fake_backend"])()
            print(f"[HW] LOCAL DRY RUN on {be.name} (Aer with that device's noise model, no QPU time used)")
            return be
        from qiskit_ibm_runtime import QiskitRuntimeService
        if IBM_API_KEY.startswith("<") or IBM_CRN.startswith("<"):
            print("[HW] QISKIT_IBM_TOKEN / QISKIT_IBM_INSTANCE not set -> trying the account saved with "
                  "QiskitRuntimeService.save_account(...)")
            service = QiskitRuntimeService(channel=IBM_CHANNEL)
        else:
            service = QiskitRuntimeService(channel=IBM_CHANNEL, token=IBM_API_KEY, instance=IBM_CRN)
        if hw["backend"]:
            be = service.backend(hw["backend"])
        else:
            be = service.least_busy(operational=True, simulator=False, min_num_qubits=hw["n_qubits"])
        st = be.status()
        print(f"[HW] backend {be.name} | {be.num_qubits} qubits | "
              f"processor {getattr(be, 'processor_type', {}) or ''} | pending jobs {st.pending_jobs}")
        return be

    def _make_primitive(self, mode_obj):
        hw = self.hw
        if hw["primitive"] == "estimator":
            from qiskit_ibm_runtime import EstimatorV2
            p = EstimatorV2(mode=mode_obj)
            p.options.default_shots = hw["shots"]
            p.options.resilience_level = hw["resilience_level"]
        else:
            from qiskit_ibm_runtime import SamplerV2
            p = SamplerV2(mode=mode_obj)
            p.options.default_shots = hw["shots"]
        if self.mode != "local":           # these options only exist on the real service
            p.options.dynamical_decoupling.enable = hw["dynamical_decoupling"]
            p.options.dynamical_decoupling.sequence_type = hw["dd_sequence"]
            p.options.twirling.enable_gates = hw["twirl_gates"]
            p.options.environment.job_tags = list(hw["job_tags"])
        return p

    def _open(self):
        if self.mode == "session":
            from qiskit_ibm_runtime import Session
            try:
                # IBM caps a session at 8 h; longer budgets open a new session when one expires
                left = max(60, min(8 * 3600, int(self.budget_s - self.used_seconds())))
                self.session = Session(backend=self.backend, max_time=left)
                self._session_t0 = time.time()
                self._session_started = False   # clock moves to the first job's start (queue wait not billed)
                self.prim = self._make_primitive(self.session)
                print(f"[HW] session opened (max_time {left} s)")
                return
            except Exception as e:      # e.g. Open Plan: sessions not allowed
                print(f"[HW] could not open a session ({type(e).__name__}: {e}) -> job mode")
                self.mode = "job"
        self.prim = self._make_primitive(self.backend)     # job mode / local fake backend

    def _close_session(self):
        if self.session is not None:
            if self._session_t0 is not None:
                self.session_seconds += time.time() - self._session_t0
            with contextlib.suppress(Exception):
                self.session.close()
        self.session, self._session_t0 = None, None

    def close(self):
        self._close_session()

    # ---- budget ----
    def used_seconds(self):
        live = time.time() - self._session_t0 if self._session_t0 is not None else 0.0
        return max(self.job_seconds, self.session_seconds + live)

    def usage_line(self):
        if self.mode == "local":
            return f"[HW] {self.n_jobs} jobs, {self.n_circuits} circuits (local dry run, 0 QPU s)"
        per_job = self.used_seconds() / max(1, self.n_jobs)
        return (f"[HW] {self.n_jobs} jobs, {self.n_circuits} circuits | QPU usage ~{self.used_seconds():.0f} s "
                f"of {self.budget_s:.0f} s budget ({self.mode} mode; job-reported {self.job_seconds:.0f} s, "
                f"~{per_job:.0f} s per job)")

    @contextlib.contextmanager
    def local(self):
        self._local += 1
        try:
            yield
        finally:
            self._local -= 1

    # ---- transpile once ----
    def isa(self, n, L):
        key = (n, L)
        if key in self._isa:
            return self._isa[key]
        qc, enc, var = qppo.build_pqc(n, L)
        names = [p.name for p in list(enc) + list(var)]          # row layout used by the trainer
        col = {nm: i for i, nm in enumerate(names)}
        if self.hw["primitive"] == "sampler":
            qc.measure_all()
        pm = generate_preset_pass_manager(optimization_level=self.hw["optimization_level"],
                                          backend=self.backend,
                                          seed_transpiler=self.hw["seed_transpiler"])
        isa = pm.run(qc)
        cols = [col[p.name] for p in isa.parameters]             # params the transpiler kept, in its order
        entry = {"isa": isa, "cols": cols}
        if self.hw["primitive"] == "estimator":
            obs = [SparsePauliOp.from_sparse_list([("Z", [q], 1.0)], num_qubits=n).apply_layout(isa.layout)
                   for q in range(n)]
            entry["obs"] = [[o] for o in obs]                    # shape (n, 1) -> broadcasts with (B,) params
        phys = isa.layout.final_index_layout() if isa.layout is not None else list(range(n))
        ops = dict(isa.count_ops())
        ops.pop("barrier", None)
        print(f"[HW] transpiled PQC (opt level {self.hw['optimization_level']}): physical qubits {phys}, "
              f"depth {isa.depth(lambda i: i.operation.name != 'barrier')}, gates {ops}")
        self._isa[key] = entry
        return entry

    # ---- run ----
    def expectations(self, n, L, rows):
        """rows: (B, n + P) in the trainer's order [x_0..x_{n-1}, theta_0..]. Returns (B, n) of <Z_q>."""
        if self.used_seconds() >= self.budget_s:
            raise QPUBudgetExceeded(self.usage_line())
        e = self.isa(n, L)
        vals = np.ascontiguousarray(rows[:, e["cols"]])
        B = rows.shape[0]
        if self.hw["primitive"] == "estimator":
            pub = (e["isa"], e["obs"], vals)
        else:
            pub = (e["isa"], vals, self.hw["shots"])
        res, job = self._run_with_retry(pub)
        self.n_jobs += 1
        self.n_circuits += B
        self.job_seconds += _job_quantum_seconds(job)
        if self.mode == "session" and not getattr(self, "_session_started", True):
            self._session_started = True
            t_run = _job_running_epoch(job)
            if t_run is not None and self._session_t0 is not None and t_run > self._session_t0:
                print(f"[HW] first job waited {t_run - self._session_t0:.0f} s in the queue (not counted)")
                self._session_t0 = t_run
        data = res[0].data
        if self.hw["primitive"] == "estimator":
            z = np.asarray(data.evs, dtype=np.float64).reshape(n, B).T
        else:
            z = np.zeros((B, n))
            shots = self.hw["shots"]
            for b in range(B):
                for key, cnt in data.meas.get_counts(loc=b).items():
                    idx = int(key, 2)
                    for q in range(n):
                        z[b, q] += cnt * (1.0 - 2.0 * ((idx >> q) & 1))
            z /= shots
        return np.clip(z, -1.0, 1.0)          # mitigated values can overshoot [-1, 1] slightly

    def _run_with_retry(self, pub, attempts=3):
        for a in range(attempts):
            try:
                job = self.prim.run([pub])
                return job.result(), job
            except Exception as ex:
                print(f"[HW] job failed ({type(ex).__name__}: {ex}); attempt {a + 1}/{attempts}")
                if a == attempts - 1:
                    raise
                if self.used_seconds() >= self.budget_s:
                    raise QPUBudgetExceeded(self.usage_line())
                if self.mode == "session":                       # session may have expired: reopen
                    self._close_session()
                    self._open()
                time.sleep(5 * (a + 1))


def _job_quantum_seconds(job):
    for get in (lambda: job.usage(), lambda: job.metrics()["usage"]["quantum_seconds"]):
        with contextlib.suppress(Exception):
            v = get()
            if v is not None:
                return float(v)
    return 0.0


def _job_running_epoch(job):
    """Unix time at which the job started running on the QPU (None if not reported)."""
    from datetime import datetime
    with contextlib.suppress(Exception):
        ts = job.metrics()["timestamps"]["running"]
        if ts:
            return datetime.fromisoformat(str(ts).replace("Z", "+00:00")).timestamp()
    return None


HW = None      # the HardwareContext, created in section 6

### 4.2 Drop-in replacement for `PQCSimulator`
Same interface (`expectations`, `grad_samples`, `rng`, `n_params`), so `QuantumLayer` and the parameter-shift autograd work unchanged. Inside `HW.local()` circuits go to the local ideal simulator instead.

In [ ]:
class HardwarePQC:
    def __init__(self, n_qubits, n_layers, opts=None):
        o = opts or {}
        self.n, self.L = n_qubits, n_layers
        self.n_params = 2 * n_qubits * n_layers
        self.grad_samples = o.get("pqc_grad_samples")
        self.rng = np.random.default_rng(int(o.get("seed", 0)))
        self._ideal = None
        HW.isa(n_qubits, n_layers)                     # transpile now, so the layout is printed up front

    def _ideal_sim(self):
        if self._ideal is None:
            self._ideal = _OrigPQCSimulator(self.n, self.L, {"noise": False})
        return self._ideal

    def expectations(self, x, theta):
        x = np.atleast_2d(np.asarray(x, dtype=np.float64))
        theta = np.asarray(theta, dtype=np.float64)
        if HW._local:
            return self._ideal_sim().expectations(x, theta)
        if theta.ndim == 1:
            theta = np.broadcast_to(theta, (x.shape[0], theta.size))
        rows = np.concatenate([x, theta], axis=1)
        return HW.expectations(self.n, self.L, rows)

### 4.3 PPO agent with fewer jobs
Same as the simulator agent, except: a rollout step runs the actor **and** critic circuits in one job (same ansatz, different parameters), and the KL early-stop uses the minibatch log-probs already computed instead of one more forward pass over the whole rollout.

In [ ]:
class HardwarePPOAgent(_OrigQuantumPPOAgent):
    @torch.no_grad()
    def select_actions(self, obs_batch):
        obs_t = torch.as_tensor(np.asarray(obs_batch), dtype=torch.float32)
        a, v = self.actor, self.value
        B = obs_t.shape[0]
        ang = torch.cat([a.pre_encoder(obs_t), v.pre_encoder_v(obs_t)]) * (np.pi / 2.0)
        th = np.concatenate([np.tile(a.pqc.theta.detach().double().numpy(), (B, 1)),
                             np.tile(v.pqc_v.theta.detach().double().numpy(), (B, 1))])
        z = torch.as_tensor(a.pqc.sim.expectations(ang.double().numpy(), th), dtype=torch.float32)
        z_a, z_v = z[:B], z[B:]
        cat = torch.distributions.Categorical(logits=a.user_head(z_a))
        mean = a.move_head(z_a)
        gauss = torch.distributions.Normal(mean, torch.exp(a.log_std).expand_as(mean))
        m, cont = cat.sample(), gauss.sample()
        logp = cat.log_prob(m) + gauss.log_prob(cont).sum(-1)
        value = v.value_head(z_v).squeeze(-1)
        if self.cfg["normalize_returns"]:
            value = value * self.ret_rms.std
        return m.numpy(), cont.numpy(), logp.numpy(), value.numpy()

    def update(self, buf, last_value):
        c = self.cfg
        nn = torch.nn
        adv, ret = self.compute_gae(np.array(buf["rewards"]), np.array(buf["values"]),
                                    np.array(buf["dones"]), last_value)
        adv, ret = adv.reshape(-1), ret.reshape(-1)
        obs_np = np.array(buf["obs"])
        obs_t = torch.as_tensor(obs_np.reshape(-1, obs_np.shape[-1]), dtype=torch.float32)
        m_t = torch.as_tensor(np.array(buf["users"]).reshape(-1), dtype=torch.long)
        cont_np = np.array(buf["conts"])
        cont_t = torch.as_tensor(cont_np.reshape(-1, cont_np.shape[-1]), dtype=torch.float32)
        logp_old = torch.as_tensor(np.array(buf["logp"]).reshape(-1), dtype=torch.float32)
        if c["normalize_returns"]:
            self.ret_rms.update(ret)
            ret_target = ret / self.ret_rms.std
        else:
            ret_target = ret
        adv_t = torch.as_tensor(adv, dtype=torch.float32)
        adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)
        ret_t = torch.as_tensor(ret_target, dtype=torch.float32)
        T = len(adv)

        a_losses, v_losses, ent_u, ent_m, kls, gq_a, gq_v = [], [], [], [], [], [], []
        for _ in range(c["policy_epochs"]):
            kl_mb = []
            for idx in self._mini_batches(T):
                cat, gauss = self.actor.dist(obs_t[idx])
                logp = cat.log_prob(m_t[idx]) + gauss.log_prob(cont_t[idx]).sum(-1)
                with torch.no_grad():   # KL of the current policy vs the rollout policy, on this minibatch
                    log_r = logp.detach() - logp_old[idx]
                    kl_mb.append(float(((torch.exp(log_r) - 1) - log_r).mean()))
                e_u = cat.entropy().mean()
                e_m = gauss.entropy().sum(-1).mean()
                ratio = torch.exp(logp - logp_old[idx])
                s1 = ratio * adv_t[idx]
                s2 = torch.clamp(ratio, 1 - c["clip_eps"], 1 + c["clip_eps"]) * adv_t[idx]
                loss = -torch.min(s1, s2).mean() - c["ent_coef_user"] * e_u - c["ent_coef_move"] * e_m
                self.actor_nn_opt.zero_grad()
                self.actor_q_opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.actor.classical_parameters(), c["max_grad_norm"])
                gq_a.append(float(nn.utils.clip_grad_norm_([self.actor.pqc.theta], c["max_grad_norm"])))
                self.actor_nn_opt.step()
                self.actor_q_opt.step()
                a_losses.append(loss.item()); ent_u.append(e_u.item()); ent_m.append(e_m.item())
            kls.append(float(np.mean(kl_mb)))
            if kls[-1] > c["target_kl"]:
                break

        for _ in range(c["value_epochs"]):
            for idx in self._mini_batches(T):
                v_pred = self.value(obs_t[idx])
                v_loss = 0.5 * nn.functional.mse_loss(v_pred, ret_t[idx])
                self.value_nn_opt.zero_grad()
                self.value_q_opt.zero_grad()
                v_loss.backward()
                nn.utils.clip_grad_norm_(self.value.classical_parameters(), c["max_grad_norm"])
                gq_v.append(float(nn.utils.clip_grad_norm_([self.value.pqc_v.theta], c["max_grad_norm"])))
                self.value_nn_opt.step()
                self.value_q_opt.step()
                v_losses.append(v_loss.item())

        return {
            "actor_loss": float(np.mean(a_losses)),
            "critic_loss": float(np.mean(v_losses)),
            "entropy_user": float(np.mean(ent_u)),
            "entropy_move": float(np.mean(ent_m)),
            "kl": kls[-1] if kls else 0.0,
            "actor_theta_grad": float(np.mean(gq_a)),
            "value_theta_grad": float(np.mean(gq_v)),
        }

## 5. Hooks into the simulator trainer
Greedy evaluation runs on the local ideal simulator unless `EVAL_ON_HARDWARE`; `run_info.json` gets a `hardware` block (backend, mode, jobs, QPU seconds, settings); the log header shows the hardware settings.

In [ ]:
def _hw_label():
    if HW.mode == "local":
        return f"local fake {HW.backend.name}"
    return f"IBM {HW.backend.name}"


def _evaluate(agent, env, seed=None):
    if HW.hw["eval_on_hardware"]:
        return _orig_evaluate(agent, env, seed)
    with HW.local():                           # greedy trajectory on the ideal simulator: no QPU time
        return _orig_evaluate(agent, env, seed)


_SAVES = [0]     # how many times this run saved a checkpoint


def _save_results(history, out_dir, cfg_train, cfg_sys, env=None, eval_summary=None, traj=None, served=None):
    _SAVES[0] += 1
    _orig_save_results(history, out_dir, cfg_train, cfg_sys, env, eval_summary, traj, served)
    path = os.path.join(out_dir, "run_info.json")
    with open(path) as f:
        info = json.load(f)
    info["algorithm"] = f"Quantum PPO (hardware: {_hw_label()})"
    info["hardware"] = {"backend": HW.backend.name, "mode": HW.mode, "jobs": HW.n_jobs,
                        "circuits": HW.n_circuits, "qpu_seconds_used": HW.used_seconds(),
                        "eval_greedy_on": "hardware" if HW.hw["eval_on_hardware"] else "ideal simulator",
                        "settings": {k: v for k, v in HW.hw.items() if k != "job_tags"}}
    with open(path, "w") as f:
        json.dump(info, f, indent=2)
    print("      " + HW.usage_line())


def _print_header(env, cfg_train, n_updates):
    c, h = env.cfg, HW.hw
    mit = f"resilience={h['resilience_level']}" if h["primitive"] == "estimator" else "no mitigation"
    print("=" * len(qppo.HEADER))
    print(f"Quantum PPO on {_hw_label()} | obs={env.state_dim} | M={c.M} qubits | VQC layers="
          f"{cfg_train['n_layers_vqc']} | {h['primitive']} {mit} | shots={h['shots']} | "
          f"opt_level={h['optimization_level']} | mode={HW.mode} | ~{n_updates} updates")
    print("=" * len(qppo.HEADER))
    print(qppo.HEADER)
    print("-" * len(qppo.HEADER))


def install_hooks():
    qppo.PQCSimulator = HardwarePQC
    qppo.QuantumPPOAgent = HardwarePPOAgent
    qppo.evaluate = _evaluate
    qppo.save_results = _save_results
    qppo.print_training_header = _print_header

## 6. Run settings

This replaces the script's command-line arguments.

| script | here |
|---|---|
| `--fake-backend FakeFez --resume results/qppo_uav_traj_model.pt --episodes 34` | free local dry run |
| `--resume results/qppo_uav_traj_model.pt --episodes 68 --max-qpu-minutes 8` | real hardware, fine-tune a simulator checkpoint |

In [ ]:
RESUME           = "results/qppo_uav_traj_model.pt"   # simulator checkpoint to fine-tune (None = from scratch)
HISTORY_DIR      = None                     # folder of the old CSVs (default: the .pt's folder)
N_EPISODES       = HW_TRAIN_OVERRIDES["n_episodes"]
OUT_DIR          = HW_TRAIN_OVERRIDES["out_dir"]
SEED             = qppo.TRAIN_CFG["seed"]

BACKEND          = HW_CFG["backend"]        # e.g. "ibm_fez" (None = least busy)
FAKE_BACKEND     = "FakeFez"                # local dry run; set to None for the real QPU
MODE             = HW_CFG["mode"]           # "session" | "job"
PRIMITIVE        = HW_CFG["primitive"]      # "estimator" | "sampler"
SHOTS            = HW_CFG["shots"]
OPT_LEVEL        = HW_CFG["optimization_level"]
RESILIENCE_LEVEL = HW_CFG["resilience_level"]
DYNAMICAL_DECOUPLING = HW_CFG["dynamical_decoupling"]
TWIRL_GATES      = HW_CFG["twirl_gates"]
MAX_QPU_MINUTES  = HW_CFG["max_qpu_minutes"]
EVAL_ON_HARDWARE = HW_CFG["eval_on_hardware"]

ROLLOUT_STEPS    = HW_TRAIN_OVERRIDES["rollout_steps"]
N_ENVS           = HW_TRAIN_OVERRIDES["n_envs"]          # more envs = fewer sequential rollout jobs
MINI_BATCH_SIZE  = HW_TRAIN_OVERRIDES["mini_batch_size"]
POLICY_EPOCHS    = HW_TRAIN_OVERRIDES["policy_epochs"]
VALUE_EPOCHS     = HW_TRAIN_OVERRIDES["value_epochs"]
GRAD_SAMPLES     = HW_TRAIN_OVERRIDES["pqc_grad_samples"]

cfg = dict(qppo.TRAIN_CFG)
cfg.update(HW_TRAIN_OVERRIDES)
cfg.update(n_episodes=N_EPISODES, out_dir=OUT_DIR, seed=SEED, shots=SHOTS,
           rollout_steps=ROLLOUT_STEPS, n_envs=N_ENVS, mini_batch_size=MINI_BATCH_SIZE,
           policy_epochs=POLICY_EPOCHS, value_epochs=VALUE_EPOCHS, pqc_grad_samples=GRAD_SAMPLES,
           resume_history_dir=HISTORY_DIR)

hw = dict(HW_CFG)
hw.update(backend=BACKEND, fake_backend=FAKE_BACKEND, mode=MODE, primitive=PRIMITIVE,
          shots=SHOTS, optimization_level=OPT_LEVEL, resilience_level=RESILIENCE_LEVEL,
          dynamical_decoupling=DYNAMICAL_DECOUPLING, twirl_gates=TWIRL_GATES,
          max_qpu_minutes=MAX_QPU_MINUTES, eval_on_hardware=EVAL_ON_HARDWARE,
          n_qubits=qppo.SystemConfig().M)
cfg["hardware"] = {k: v for k, v in hw.items() if k != "job_tags"}   # saved into the .pt and run_info.json

### Train
Stops cleanly at the QPU budget or on interrupt (Kernel → Interrupt); the last finished update is always saved.

In [ ]:
if not RESUME:
    print("[HW] WARNING: no RESUME checkpoint. Training from scratch on a QPU costs many QPU-hours; "
          "the intended use is fine-tuning a simulator checkpoint.")

HW = HardwareContext(hw)
install_hooks()
_SAVES[0] = 0
try:
    agent, history = qppo.train(cfg, resume=RESUME)
except QPUBudgetExceeded as e:
    print(f"\n[HW] QPU budget reached, stopped before the next job. {e}")
    if _SAVES[0] == 0:
        print("[HW] No update finished in this run, so nothing new was saved: the budget ran out during the "
              "first update. Raise MAX_QPU_MINUTES, or lower the jobs per update (N_ENVS, ROLLOUT_STEPS).")
    else:
        print(f"[HW] {_SAVES[0]} update(s) saved in {os.path.abspath(OUT_DIR)} (the update in progress was "
              f"dropped); continue later with RESUME = {os.path.join(OUT_DIR, 'qppo_uav_traj_model.pt')!r}")
except KeyboardInterrupt:
    print("\n[HW] interrupted; the last finished update is saved in", os.path.abspath(OUT_DIR))
finally:
    HW.close()
    print(HW.usage_line())

## 7. Results

In [ ]:
from IPython.display import Image, display
for name in ["quantum_ppo_training.png", "trajectory.png"]:
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(name); display(Image(filename=path))
    else:
        print(f"[missing] {path}")
run = qppo.load_results(OUT_DIR)
print(run["info"].get("algorithm"), "|", run["info"].get("n_episodes_done"), "episodes |",
      run["info"].get("n_updates_done"), "updates")
print("hardware:", json.dumps(run["info"].get("hardware"), indent=2))